In [1]:
import os
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix

In [2]:
bow = pd.read_csv("../data/output/BOW.csv", sep="|")

print("BOW shape:", bow.shape)
bow.head()

BOW shape: (616248, 7)


,book_number,chapter,verse,term_str,n,idf,tfidf
0,1,1,1,and,1,0.264775,0.264775
1,1,1,1,beginning,1,5.700637,5.700637
2,1,1,1,created,1,6.707441,6.707441
3,1,1,1,earth,1,3.535988,3.535988
4,1,1,1,god,1,2.082210,2.082210


In [3]:
bag_cols = ["book_number", "chapter", "verse"]

bow["doc_id"] = (
    bow["book_number"].astype(str) + "_" +
    bow["chapter"].astype(str) + "_" +
    bow["verse"].astype(str)
)

In [4]:
doc_index = {doc:i for i, doc in enumerate(bow["doc_id"].unique())}
term_index = {term:i for i, term in enumerate(bow["term_str"].unique())}

num_docs = len(doc_index)
num_terms = len(term_index)

print("Number of documents:", num_docs)
print("Number of terms:", num_terms)

Number of documents: 31102
Number of terms: 12878


In [5]:
rows = bow["doc_id"].map(doc_index)
cols = bow["term_str"].map(term_index)
data = bow["n"]

dtm = csr_matrix((data, (rows, cols)), shape=(num_docs, num_terms))

print("DTM shape:", dtm.shape)

DTM shape: (31102, 12878)


In [8]:
print(dtm[:5].toarray())

[[1 1 1 ... 0 0 0]
 [4 0 0 ... 0 0 0]
 [2 0 0 ... 0 0 0]
 [2 0 0 ... 0 0 0]
 [4 0 0 ... 0 0 0]]


In [9]:
sample = pd.DataFrame(
    dtm[:5].toarray(),
    columns=list(term_index.keys())
)

sample.iloc[:, :10]  # first 10 terms

,and,beginning,created,earth,god,heaven,in,the,darkness,deep
0,1,1,1,1,1,1,1,3,0,0
1,4,0,0,1,1,0,0,6,1,1
2,2,0,0,0,1,0,0,0,0,0
3,2,0,0,0,2,0,0,3,1,0
4,4,0,0,0,1,0,0,5,1,0


In [10]:
print("Non-zero entries:", dtm.nnz)

density = dtm.nnz / (dtm.shape[0] * dtm.shape[1])
print("Matrix density:", density)

Non-zero entries: 616248
Matrix density: 0.0015385754025333276


In [11]:
delimiter = "|"
print("Delimiter:", delimiter)

bag_string = " | ".join(bag_cols)
print("Bag (expressed in terms of OHCO levels):", bag_string)

Delimiter: |
Bag (expressed in terms of OHCO levels): book_number | chapter | verse


In [12]:
from scipy.sparse import save_npz

os.makedirs("../data/output", exist_ok=True)

save_npz("../data/output/DTM.npz", dtm)

print("DTM saved.")

DTM saved.
